In [2]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv("retail_sales.csv")

df.head()

,Order_ID,Order_Date,Customer_ID,Product,Category,Quantity,Price,Total_Sales,City,Gender
0,ORD0001,2025-05-21,CUST001,Office Chair,Furniture,2,7500,15000,Mumbai,Male
1,ORD0002,2025-10-07,CUST022,Office Chair,Furniture,1,7500,7500,Hyderabad,Female
2,ORD0003,2025-04-22,CUST003,Laptop,Electronics,2,65000,130000,Hyderabad,Male
3,ORD0004,2025-11-29,CUST023,Desk,Furniture,5,6000,30000,Bangalore,Male
4,ORD0005,2025-01-04,CUST009,Dress,Clothing,2,2200,4400,Chennai,Female


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 200
Columns: 10
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Order_ID     200 non-null    object
 1   Order_Date   200 non-null    object
 2   Customer_ID  200 non-null    object
 3   Product      200 non-null    object
 4   Category     200 non-null    object
 5   Quantity     200 non-null    int64 
 6   Price        200 non-null    int64 
 7   Total_Sales  200 non-null    int64 
 8   City         200 non-null    object
 9   Gender       200 non-null    object
dtypes: int64(3), object(7)
memory usage: 15.8+ KB


In [5]:
df.isnull().sum()

Order_ID       0
Order_Date     0
Customer_ID    0
Product        0
Category       0
Quantity       0
Price          0
Total_Sales    0
City           0
Gender         0
dtype: int64

In [6]:
conn = sqlite3.connect("retail_sales.db")

df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

print("Data successfully loaded into SQLite!")

Data successfully loaded into SQLite!


In [7]:
query = """
SELECT SUM(Total_Sales) AS Total_Revenue
FROM sales;
"""

pd.read_sql_query(query, conn)

,Total_Revenue
0,6138700


In [8]:
query = """
SELECT
    Category,
    SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Category
ORDER BY Revenue DESC;
"""

pd.read_sql_query(query, conn)

,Category,Revenue
0,Electronics,3460200
1,Furniture,2229000
2,Clothing,449500


In [9]:
query = """
SELECT
    Product,
    SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Product
ORDER BY Revenue DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Product,Revenue
0,Laptop,1950000
1,Sofa,1050000
2,Smartphone,870000
3,Monitor,408000
4,Office Chair,397500
5,Table,324000
6,Desk,282000
7,Bookshelf,175500
8,Jacket,143500
9,Headphones,135000


In [10]:
query = """
WITH monthly_sales AS (
    SELECT
        strftime('%Y-%m', Order_Date) AS Month,
        SUM(Total_Sales) AS Revenue
    FROM sales
    GROUP BY Month
)

SELECT
    Month,
    Revenue,
    LAG(Revenue) OVER (ORDER BY Month) AS Previous_Month_Revenue,
    Revenue - LAG(Revenue) OVER (ORDER BY Month) AS Revenue_Change
FROM monthly_sales
ORDER BY Month;
"""

pd.read_sql_query(query, conn)

,Month,Revenue,Previous_Month_Revenue,Revenue_Change
0,2025-01,863700,NaN,NaN
1,2025-02,464800,863700.0,-398900.0
2,2025-03,724300,464800.0,259500.0
3,2025-04,658800,724300.0,-65500.0
4,2025-05,158600,658800.0,-500200.0
5,2025-06,240800,158600.0,82200.0
6,2025-07,438000,240800.0,197200.0
7,2025-08,488400,438000.0,50400.0
8,2025-09,546400,488400.0,58000.0
9,2025-10,760900,546400.0,214500.0


In [11]:
customers = df[
    ["Customer_ID", "City", "Gender"]
].drop_duplicates("Customer_ID")

customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("Customers table created!")

Customers table created!


In [12]:
query = """
SELECT
    c.City,
    SUM(s.Total_Sales) AS Revenue
FROM sales s
JOIN customers c
    ON s.Customer_ID = c.Customer_ID
GROUP BY c.City
ORDER BY Revenue DESC;
"""

pd.read_sql_query(query, conn)

,City,Revenue
0,Mumbai,1347500
1,Pune,1127200
2,Chennai,1080200
3,Delhi,1037300
4,Hyderabad,791000
5,Bangalore,755500


In [13]:
query = """
SELECT
    AVG(Total_Sales) AS Average_Order_Value
FROM sales;
"""

pd.read_sql_query(query, conn)

,Average_Order_Value
0,30693.5


In [14]:
query = """
SELECT
    Customer_ID,
    COUNT(Order_ID) AS Number_of_Orders,
    SUM(Total_Sales) AS Total_Spent
FROM sales
GROUP BY Customer_ID
ORDER BY Total_Spent DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Customer_ID,Number_of_Orders,Total_Spent
0,CUST018,9,495300
1,CUST019,7,488400
2,CUST024,8,465500
3,CUST011,6,376400
4,CUST029,7,362000
5,CUST030,6,324000
6,CUST020,11,298000
7,CUST003,12,276900
8,CUST028,7,272100
9,CUST009,9,229500


In [15]:
query = """
WITH product_sales AS (
    SELECT
        Product,
        SUM(Total_Sales) AS Revenue
    FROM sales
    GROUP BY Product
)

SELECT
    Product,
    Revenue,
    RANK() OVER (ORDER BY Revenue DESC) AS Revenue_Rank
FROM product_sales
ORDER BY Revenue_Rank;
"""

pd.read_sql_query(query, conn)

,Product,Revenue,Revenue_Rank
0,Laptop,1950000,1
1,Sofa,1050000,2
2,Smartphone,870000,3
3,Monitor,408000,4
4,Office Chair,397500,5
5,Table,324000,6
6,Desk,282000,7
7,Bookshelf,175500,8
8,Jacket,143500,9
9,Headphones,135000,10


In [16]:
query = """
SELECT
    strftime('%Y-%m', Order_Date) AS Month,
    Category,
    SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Month, Category
ORDER BY Month, Revenue DESC;
"""

pd.read_sql_query(query, conn)

,Month,Category,Revenue
0,2025-01,Electronics,492400
1,2025-01,Furniture,303000
2,2025-01,Clothing,68300
3,2025-02,Furniture,218500
4,2025-02,Electronics,203200
5,2025-02,Clothing,43100
6,2025-03,Furniture,466500
7,2025-03,Electronics,219400
8,2025-03,Clothing,38400
9,2025-04,Electronics,382000


In [17]:
query = """
SELECT
    Product,
    SUM(Quantity) AS Units_Sold,
    SUM(Total_Sales) AS Revenue
FROM sales
GROUP BY Product
ORDER BY Units_Sold DESC
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,Product,Units_Sold,Revenue
0,Keyboard,54,97200
1,Dress,54,118800
2,Office Chair,53,397500
3,Desk,47,282000
4,Headphones,45,135000
5,Sofa,42,1050000
6,Jacket,41,143500
7,Bookshelf,39,175500
8,Table,36,324000
9,T-Shirt,35,28000
